[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/geometry/cyclides/cyclides.ipynb)

# Dupin Cyclides on the 3-Sphere

This notebook ray traces Dupin cyclides in a curved space, the 3-sphere S³: the unit sphere in four dimensions, seen from inside. Light travels along great circles, the camera and every placement are rotations of R⁴, and there is no point at infinity. The starting point is the spherical cylinder, a tube around a great circle and straight along its core. A conformal dilation bends that core into a small circle, and the tube becomes a Dupin cyclide.

The surfaces are quadrics in conformal geometric algebra. Each pixel's ray is substituted into the surface equation, which gives a polynomial whose nearest root is the hit.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from numga import Algebra, NumpyContext, stack
from examples.animation import save_animation
from examples.geometry.cyclides import render
from examples.geometry.cyclides.core import pixel_grid, nearest_angle

# The conformal model of S³: the four directions x, y, z, w of R⁴, plus one direction e that squares to -1.
# A point p of S³ is a unit vector of R⁴; it is carried as p + e, which squares to 1 - 1 = 0: a null vector.
ga = Algebra("x+y+z+w+e-")
ctx = NumpyContext(ga)
mv = ctx.multivector

Scalar = ga.gatype.scalar()
Sphere = ga.gatype.vector()                   # Grade 1: a sphere is a vector s; the points on it are those with s . (p + e) == 0
Point = ga.gatype.antivector()                # Grade 4: a point, as the dual of its null vector, so that `sphere & point` is that inner product
Direction = ga.gatype.from_blades("x y z")    # the directions at the eye: the eye sits at w, and x, y, z span the tangent space there
Quadric = ga.gatype((Sphere, Point))          # a quadric as a map: point -> its polar sphere; the point is on the surface when it lies on its own polar
Motor = ga.gatype.rotor()                     # rotations of R⁴, dilations, and their products: the conformal motions of S³

origin: Point = (mv.w + mv.e).dual()          # the eye, at w on S³
antipode: Point = (-mv.w + mv.e).dual()       # the point opposite the eye, at -w

# The image: one unit direction per pixel, for a pinhole camera at the eye looking along -x with a 90 degree view.
shape = (240, 320)
pixels: Direction = mv(Direction, pixel_grid(shape, np.radians(90))).normalized()   # [n_pixels] Direction

np.set_printoptions(precision=6, suppress=True)

## 1. Points and Spheres

A point of S³ is a unit vector of R⁴, carried as a null vector with one unit of `e` added. A sphere of angular radius `r` about a point is that point's unit vector plus `cos(r)` units of `e`; at a quarter turn the `e` part vanishes, and the sphere is a great sphere. The regressive product `sphere & point` is zero exactly when the point lies on the sphere. `e & point` is the same for every point, and `e` itself is not a point of S³.

In [ ]:
# A sphere of angular radius r about a point c is c + e cos(r): against a point p + e it gives c . p - cos(r),
# which vanishes exactly when the angle between c and p is r. Here c is the eye itself, w.
radius = 0.4
sphere: Sphere = mv.w + mv.e * np.cos(radius)                                # [] Sphere
# Turn the eye by 0.4 towards x: the rotor exp(0.5 * angle * (x ^ w)) turns by `angle` in the xw plane.
turned: Point = (0.5 * radius * (mv.x ^ mv.w)).exp() >> origin               # [] Point

# The eye itself is 0 rad from the centre, not 0.4, so it is off the sphere; the turned eye is on it.
# Every point carries exactly one unit of e, so e & point is the same for all of them: e is not a point of S³.
incidence: Scalar = stack((sphere & origin, sphere & turned, mv.e & origin, mv.e & turned))   # [incidences] Scalar

print("sphere & eye, sphere & turned eye, e & eye, e & turned eye:", incidence.to_array())

## 2. A Ray Is a Great Circle

Turning the eye towards a direction moves it along a great circle. The generator of that turn squares to a negative number, so its exponential sums to a cosine and a sine. The coefficient maps are commutators with the generator: the first and second derivatives of the turning eye.

In [ ]:
# The inner product of two directions, left open in both: d | d is 1 for a unit direction.
metric = Direction | Direction                                               # [] Scalar <- (Direction, Direction)
# The generator of the turn towards d: a rotation in the plane of d and w. It squares to -1 for a unit d,
# and the factor 0.5 makes exp(t * ray_rotation(d)) turn the eye by exactly t.
ray_rotation = 0.5 * (Direction ^ mv.w)                                      # [] Bivector <- Direction
# Derivatives of the turning eye at t = 0: each is a commutator with the generator (the factor 2 undoes the
# commutator's 1/2). The first is the eye's velocity, heading off towards d; the second its acceleration.
# ray_rotation and ray_linear each bring their own open Direction, so their commutator has two slots, in order:
ray_linear = ray_rotation.commutator(origin) * 2                             # [] Point <- Direction
ray_quadratic = ray_rotation.commutator(ray_linear) * 2                      # [] Point <- (Direction, Direction)
# Since the generator squares to -1, every higher derivative repeats these two up to sign, and the whole series
# sums to a cosine and a sine. The eye turned by t towards a unit d:
#     origin + sin(t) * ray_linear(d) + (1 - cos(t)) * ray_quadratic(d, d)

# Direction is the part x y z of the vectors, and a point is an antivector, so the printed type reads
# in antivector <- in vector, in vector: a point from two directions.
print(ray_quadratic)

In [ ]:
# checks: follow one unit direction to two angles, once with the exact rotor and once with the formula above,
# and test both results against the spheres of radius t about the eye.
direction = mv(Direction, [0.0, 0.6, 0.8])                                   # [] Direction
angles = np.array([0.7, 2.5])
exact: Point = (ray_rotation(direction) * angles).exp() >> origin            # [angles] Point
circle: Point = origin + ray_linear(direction) * np.sin(angles) + ray_quadratic(direction, direction) * (1 - np.cos(angles))
reach: Sphere = mv.w + mv.e * np.cos(angles)                                 # [angles] Sphere: radius t about the eye
on_reach: Scalar = stack((reach & exact, reach & circle), axis=-1)           # [angles, methods] Scalar: all zero

print("exact and circle points on the spheres of radius t:\n", on_reach.to_array())

Points are only defined up to scale. Multiplied by `1 + u**2`, with the half angle `u = tan(t / 2)`, the circle becomes a parabola in `u`, and its bend is the antipode of the eye.

In [ ]:
# With u = tan(t / 2): sin(t) = 2u / (1 + u**2) and 1 - cos(t) = 2u**2 / (1 + u**2). Multiplying the turned eye by
# 1 + u**2, which does not move a point, clears the denominators:
#     (1 + u**2) * eye(t) == origin + 2u * ray_linear(d) + u**2 * (origin + 2 * ray_quadratic(d, d))
# The u**2 term is the bend. `metric` writes the origin in it as (d | d) * origin, so the bend is quadratic in d like
# the rest of it; for a unit d that is just the origin. As u grows the bend dominates: t = pi reaches the antipode.
ray_bend = origin * metric + ray_quadratic * 2                               # [] Point <- (Direction, Direction)

In [ ]:
# checks: the bend is the antipode, and the parabola, divided by 1 + u**2 again, lands on the circle.
# Five probes, the basis spheres, read every coefficient of a point: vanishing against all five means equal.
u = np.tan(angles / 2)
parabola: Point = origin + ray_linear(direction) * (2 * u) + ray_bend(direction, direction) * u**2   # [angles] Point
probes: Sphere = stack((mv.x, mv.y, mv.z, mv.w, mv.e))                      # [probes] Sphere
bend_gap: Scalar = probes & (ray_bend(direction, direction) - antipode)     # [probes] Scalar
parabola_gap: Scalar = probes[:, None] & (parabola / (1 + u**2) - circle)   # [probes, angles] Scalar

print("bend - antipode against five spheres:", bend_gap.to_array())
print("parabola / (1 + u**2) - circle:\n", parabola_gap.to_array())

## 3. Tracing the Spherical Cylinder

A quadric is a sum of dyads, one per sphere, with weights. As an extensor `Sphere <- Point` it sends a point to its polar sphere, and the point lies on the surface when it is incident with that polar. Each dyad contributes the squared incidence with its sphere.

The simplest one to trace is the spherical cylinder: all points at a fixed angle from a great circle, straight along its core. Two great spheres and `e` build it.

In [ ]:
# Each dyad s * (s & Point) sends a point to s, weighted by its incidence with s; paired with the point again it gives
# that incidence squared. For p + e: z**2 + w**2 - sin(tube)**2, since e & (p + e) is the same unit for every point.
# The great circle in the xy plane is where z = w = 0, and z**2 + w**2 is sin**2 of the angle from it, so the zero set
# is every point at angle `tube` from that circle: a tube around a great circle, straight along its core.
tube = 0.3
cylinder: Quadric = mv.z * (mv.z & Point) + mv.w * (mv.w & Point) - np.sin(tube)**2 * mv.e * (mv.e & Point)   # [] Sphere <- Point

# Place it with rotations of R⁴, which are the rigid motions of S³: turning in yz stands its core upright in the
# image, and turning in xw carries it towards the view direction, to within 1.2 rad of the eye.
place = (mv.xw * ((np.pi / 2 - 1.2) / 2)).exp() * (mv.yz * ((np.pi / 2 - 0.3) / 2)).exp()   # [] Motor
# A quadric moves as a sandwich on both sides: its input point is pulled back, its output sphere pushed forward.
placed: Quadric = (place >> cylinder(place << Point)).reshape(1)             # [surfaces] Sphere <- Point

To trace, put each pixel's ray into the surface equation. The eye stays at `w`, so camera coordinates are world coordinates and the camera map is the identity. Along the parabola in `u` the equation is a polynomial of degree four in `u`, with one coefficient form per power and one direction slot per power.

`nearest_angle` solves for `u` and returns the smallest angle along the whole circle, so a ray that passes the antipode keeps going. At the hit, the polar sphere is the tangent sphere, and its length is the length of the surface gradient: the gradient along the ray's unit velocity over that length is the cosine between the ray and the normal. The function below does this for a batch of surfaces, and the later sections reuse it.

In [ ]:
def trace(surfaces: Quadric, pixels: Direction):
    # The quadric as a symmetric form on points: form(X, X) = X & Q(X), a point against its own polar sphere,
    # zero exactly on the surface. The eye stays at w, so the points are world points throughout.
    form = Point & surfaces                                                  # [n] Scalar <- (Point, Point)

    # Put the ray X = origin + 2u ray_linear(d) + u**2 ray_bend(d, d) into form(X, X). A map passed into a slot keeps
    # its own open slots, so each term keeps one Direction slot per factor of d: ray_linear brings one, ray_bend two.
    # The form is symmetric, so the cross terms pair up, and each power of u collects its own form in d. These five
    # maps are the ray polynomial for every direction at once, built before any pixel is seen:
    constant = form(origin, origin)                                          # [n] Scalar: the eye against the surface
    linear = 4 * form(origin, ray_linear)                                    # [n] Scalar <- Direction
    quadratic = 4 * form(ray_linear, ray_linear) + 2 * form(origin, ray_bend)   # [n] Scalar <- (Direction, Direction)
    cubic = 4 * form(ray_linear, ray_bend)                                   # [n] Scalar <- (Direction, Direction, Direction)
    quartic = form(ray_bend, ray_bend)                                       # [n] Scalar <- (Direction, Direction, Direction, Direction)
    # The u**4 coefficient is the first map here with four open slots: ray_bend's two, twice. Nothing about four is
    # special. An extensor has as many slots as its expression leaves open, each is bound like any other, and
    # quartic(d, d, d, d) is the u**4 coefficient of the ray along d. In index notation it would be a symmetric
    # tensor contracted four times with d, Q_ijkl d^i d^j d^k d^l.

    # Tracing is now only binding: every pixel direction into every open slot gives one quartic in u per surface and
    # pixel; [:, None] puts the surfaces on the first axis and the pixels on the second. nearest_angle turns each real root u into the
    # angle t = 2 atan(u) and keeps the smallest positive one along the whole circle, past the antipode too.
    angle = nearest_angle(
        constant[:, None],
        linear[:, None](pixels),
        quadratic[:, None](pixels, pixels),
        cubic[:, None](pixels, pixels, pixels),
        quartic[:, None](pixels, pixels, pixels, pixels))                    # [n, n_pixels] t, inf on a miss
    t = np.where(np.isfinite(angle), angle, 0.0)
    # The hit, and the ray's velocity there: the derivative in t of the turned eye, of unit speed.
    hit = origin + ray_linear(pixels) * np.sin(t) + ray_quadratic(pixels, pixels) * (1 - np.cos(t))   # [n, n_pixels] Point
    velocity = ray_linear(pixels) * np.cos(t) + ray_quadratic(pixels, pixels) * np.sin(t)          # [n, n_pixels] Point

    # At a point of the surface its polar sphere is the tangent sphere, and it grows with the surface's gradient.
    # The velocity against it, over its length, is the cosine between the ray and the normal: a headlight.
    polar = surfaces[:, None](hit)                                           # [n, n_pixels] Sphere
    facing = -(polar & velocity) / (polar | polar).abs().square_root()       # [n, n_pixels] Scalar, 1 when facing the eye
    return facing, angle

facing, angle = trace(placed, pixels)
render.draw_facing(facing, angle, shape);

## 4. Bending the Core: Tori

A dilation, the exponential of `toward ^ e` for a point `toward`, is a conformal map of S³ that pushes points towards `toward` and keeps it and its antipode fixed. It acts on a quadric as a sandwich, on the output sphere and on the input point. Dilating towards `z`, a quarter turn from every point of the core, bends the core evenly into a smaller circle, and the cylinder becomes a torus. Seen from `z` a dilation is a uniform scaling, so it only sizes the torus; the tube sets its shape.

In [ ]:
# Turn a point `toward` onto the point `ahead` rad in front of the eye, after a tilt that keeps it fixed.
def placement(toward: Sphere, ahead: float, tilt: float):
    target = mv.w * np.cos(ahead) - mv.x * np.sin(ahead)                     # the point `ahead` rad along the view direction -x
    # 1 + target * toward, normalized, is the rotor that turns `toward` straight onto `target`. The tilt turns in the
    # yw plane, which leaves z and x alone, so for toward = z it only spins the shape about its aim.
    return (1 + target * toward).normalized() * (mv.yw * (tilt / 2)).exp()   # [] Motor

# Tubes of 0.15, 0.5 and 0.9 rad about the core, all the same shape as the cylinder above but for their width:
tubes = np.array([0.15, 0.5, 0.9])
tubes_around_core: Quadric = (mv.z * (mv.z & Point) + mv.w * (mv.w & Point)
                              - np.sin(tubes)**2 * mv.e * (mv.e & Point))   # [tori] Sphere <- Point
# The dilation towards z: (z ^ e) squares to +1, so its exponential is hyperbolic, cosh and sinh, rather than a rotation. It keeps z
# and -z and pushes every other point along its great circle from -z towards z. The core, a quarter turn from z,
# shrinks into a small circle about z; each strength is chosen so that its torus fills the frame.
# In classical conformal geometry, this and every other motion used here is a Möbius transformation of S³.
strengths = np.array([1.55, 2.0, 2.4])
dilations = ((mv.z ^ mv.e) * (strengths / 2)).exp()                          # [tori] Motor
views = placement(mv.z, 1.1, 0.8)                                            # [] Motor: bring z, the torus's centre, into view
place = views * dilations                                                    # [tori] Motor: first dilate, then place
tori: Quadric = place >> tubes_around_core(place << Point)                   # [tori] Sphere <- Point

facing, angle = trace(tori, pixels)
render.draw_facing(facing, angle, shape);

## 5. Dupin Cyclides

Aiming the dilation at a point near the core breaks the spin symmetry. The tube is compressed on the side of the aim and swells on the other: the lopsided Dupin cyclides.

In [ ]:
# The aim leans from z towards x, a point on the core: the dilation now pushes one side of the tube harder than the
# other. Tube radius, dilation strength, and the lean, per cyclide:
tubes = np.array([0.25, 0.3, 0.2])
strengths = np.array([1.5, 1.2, 1.8])
leans = np.array([0.7, 0.8, 0.5])
aims: Sphere = mv.z * np.cos(leans) + mv.x * np.sin(leans)                   # [cyclides] Sphere: points of S³ as unit vectors
tubes_around_core: Quadric = (mv.z * (mv.z & Point) + mv.w * (mv.w & Point)
                              - np.sin(tubes)**2 * mv.e * (mv.e & Point))   # [cyclides] Sphere <- Point
dilations = ((aims ^ mv.e) * (strengths / 2)).exp()                          # [cyclides] Motor
lopsided: Quadric = dilations >> tubes_around_core(dilations << Point)       # [cyclides] Sphere <- Point
views = placement(aims, np.array([1.2, 1.3, 1.1]), np.array([-0.7, 0.6, 1.0]))   # [cyclides] Motor: the aim into view
placed_lopsided: Quadric = views >> lopsided(views << Point)                 # [cyclides] Sphere <- Point

facing, angle = trace(placed_lopsided, pixels)
render.draw_facing(facing, angle, shape);

## 6. The Conical Case: Vertices, Lemons and Bananas

A spherical cone is the set of points whose geodesic from a vertex makes a fixed angle with an axis. It is built from two great spheres and `e`, like the cylinder. Its geodesics from the vertex meet again at the vertex's antipode, so the cone has two vertices. A strong dilation towards the axis point, a quarter turn from both vertices, pulls the vertices together and closes the cone into a spindle cyclide: two conical points joined by an outer and an inner sheet. Leaning the dilation off the axis bends the inner sheet into a banana, seen through an opening next to one vertex.

In [ ]:
# A cone with vertex z, its axis towards x, half-opening 0.35 rad. Against p + e the form reads
# x**2 + cos(opening)**2 * (z**2 - 1); on S³, 1 - z**2 = x**2 + y**2 + w**2, so the zero set is
# x**2 == cos(opening)**2 * (x**2 + y**2 + w**2): the geodesics leaving z at angle `opening` from the x axis.
# They all meet again at -z, the cone's second vertex.
opening = 0.35
cone: Quadric = mv.x * (mv.x & Point) + np.cos(opening)**2 * (mv.z * (mv.z & Point) - mv.e * (mv.e & Point))   # [] Sphere <- Point

# A strong dilation towards the axis point x, a quarter turn from both vertices, pulls the vertices towards each
# other; the second aim leans 0.7 rad towards y and bends the inner sheet.
leans = np.array([0.0, 0.7])
aims: Sphere = mv.x * np.cos(leans) + mv.y * np.sin(leans)                   # [spindles] Sphere: points of S³ as unit vectors
dilations = ((aims ^ mv.e) * (3.0 / 2)).exp()                                # [spindles] Motor
spindles: Quadric = dilations >> cone(dilations << Point)                    # [spindles] Sphere <- Point

# Where the eye should aim: halfway between the two dilated vertices. The vertices are the null vectors z + e and
# -z + e carried by the dilation; scaled to one unit of e each, their R⁴ parts are points of S³, and the
# normalized sum of those is the midpoint of the geodesic between them.
vertices: Sphere = dilations[:, None] >> stack((mv.z + mv.e, -mv.z + mv.e))  # [spindles, vertices] Sphere: null vectors
unit_e = vertices / -(vertices | mv.e)                                       # [spindles, vertices] Sphere: one unit of e each
midpoints: Sphere = (unit_e[:, 0] + unit_e[:, 1] - 2 * mv.e).normalized()    # [spindles] Sphere: drop the e parts, normalize

In [ ]:
# A rotation of R⁴ from six angles, one per coordinate plane, applied in turn.
def orientation(angles):
    rotor = mv.rotor()
    for plane, angle in zip((mv.xy, mv.xz, mv.xw, mv.yz, mv.yw, mv.zw), angles):
        rotor = rotor * (plane * angle).exp()
    return rotor                                                             # [] Motor

def view(centre: Sphere, angles, ahead: float, yaw: float, pitch: float):
    # Turn the shape about the origin of R⁴, carry its centre to `ahead` rad in front of the eye, then turn the eye.
    turned = orientation(angles)
    target = mv.w * np.cos(ahead) - mv.x * np.sin(ahead)
    carry = (1 + target * (turned >> centre)).normalized()                   # the turned centre straight onto the target
    # Rotations in the xy and xz planes leave w, the eye, where it is: they turn the view, yaw and pitch.
    look = (mv.xy * (yaw / 2)).exp() * (mv.xz * (pitch / 2)).exp()
    return look * carry * turned                                             # [] Motor

# Five shots: which spindle, its orientation in the six planes, how far ahead, and the eye's yaw and pitch.
shots = [
    (0, [-0.186, -0.692, 2.658, 0.181, -0.282, -0.225], 1.10, -0.021, 0.061),
    (0, [1.124, 0.598, 0.155, 0.889, -0.164, -0.741], 1.05, -0.051, -0.062),
    (0, [0.464, 0.073, 0.536, -2.263, 0.817, -0.768], 1.05, -0.001, 0.060),
    (1, [-0.257, 0.18, 0.46, -0.999, -1.384, -0.004], 0.85, 0.137, -0.377),
    (1, [-0.511, -0.64, -0.64, 1.096, -1.168, -0.477], 0.85, -0.297, -0.358),
]
views = stack([view(midpoints[i], angles, ahead, yaw, pitch) for i, angles, ahead, yaw, pitch in shots])   # [shots] Motor
shot_surfaces: Quadric = stack([spindles[i] for i, *_ in shots])            # [shots] Sphere <- Point
placed_shots: Quadric = views >> shot_surfaces(views << Point)               # [shots] Sphere <- Point

# Through the opening the eye sees both sides of the sheets, so the headlight takes the unsigned cosine.
facing, angle = trace(placed_shots, pixels)
render.draw_facing(facing.abs(), angle, shape);

## 7. Symmetry Instead of Degree

The cylinder is built from great spheres and `e` only. That makes its polynomial in `u` palindromic: the `u**4` and constant coefficients agree, and the `u**3` and `u` coefficients are opposite. A palindromic quartic is a quadratic in `cot(t)`, and the four hits along a great circle come in antipodal pairs. The lopsided dilation breaks that symmetry.

The degree drops only for surfaces through the antipode of the eye, where the `u**4` coefficient vanishes.

In [ ]:
# The cylinder and a Dupin cyclide, both placed, as forms on points:
surfaces: Quadric = stack((placed[0], placed_lopsided[0]))                  # [surfaces] Sphere <- Point
form = Point & surfaces                                                      # [surfaces] Scalar <- (Point, Point)

# A few pixel directions. The quartic in u is palindromic when its u**4 and constant coefficients agree and its
# u**3 and u coefficients are opposite; the two gaps below are zero exactly then. Swapping u for -1/u, which moves
# the point along the circle to its antipode, then maps roots to roots: the hits come in antipodal pairs.
sample = pixels[::20000]                                                     # [samples] Direction
outer_gap: Scalar = form(ray_bend, ray_bend)[:, None](sample, sample, sample, sample) - form(origin, origin)[:, None]   # [surfaces, samples] Scalar: u**4 - constant
odd_gap: Scalar = (4 * form(ray_linear, ray_bend))[:, None](sample, sample, sample) + (4 * form(origin, ray_linear))[:, None](sample)   # [surfaces, samples] Scalar: u**3 + u

print("u**4 - constant, for the cylinder and a Dupin cyclide:\n", outer_gap.to_array())
print("u**3 + u:\n", odd_gap.to_array())

## 8. Rolling a Torus Around a Circle

A circle is the meet of two spheres, a bivector, and its exponential turns points around the circle, the way a vortex ring carries fluid around its core. Around the torus's own core circle the flow only spins the tube in place and leaves the torus unchanged. Around that core tilted a little, one turn rolls and deforms the torus.

In [ ]:
# A circle is the meet of two spheres, their wedge: the core of every torus above is z ^ w, the great circle where
# both great spheres z and w meet. Carry it with the middle torus's own placement, after a small tilt of 0.4 rad in
# the yw plane, so that it no longer lines up with the torus's core. The versors are unit, so the circle stays a
# unit circle: circle * circle == -1.
circle = (place[1] * (mv.yw * 0.2).exp()) >> (mv.z ^ mv.w)                  # [] Bivector

# exp(circle * phase / 2) turns every point of S³ around that circle by `phase`, fixing the circle itself: points
# near it circle it quickly, like fluid around a vortex ring. One full turn over 36 frames, applied to the torus as
# a sandwich like any other motion; each frame is traced only when the animation asks for it.
phases = np.linspace(0.0, 2 * np.pi, 36, endpoint=False)
flow = (circle * (phases / 2)).exp()                                          # [frames] Motor
rolled: Quadric = flow >> tori[1](flow << Point)                              # [frames] Sphere <- Point
traced = (trace(surface.reshape(1), pixels) for surface in rolled)

display(Image(filename=save_animation(render.facing_frames(traced, shape), "cyclides_vortex", 60)))

## 9. A Linked Vortex

The flat conformal model is one chart of S³, the stereographic projection from the antipode of the eye: the eye's null vector is its origin and the antipode's is its point at infinity. Flat constructions written with these two draw the same surfaces on S³. A hyperboloid of one sheet, inverted in a sphere outside it, closes its two ends in a single conical tip; a circle threading its opening turns it around, linked with a thin ring on that circle.

In [ ]:
# The flat chart about the eye: its origin, and its point at infinity.
chart_origin: Sphere = (mv.w + mv.e) * 0.5                                  # the eye's null vector
chart_infinity: Sphere = mv.e - mv.w                                         # the antipode's null vector

def translator(displacement):
    # A translation of the chart: on S³ a conformal map that keeps the antipode fixed.
    return (-0.5 * (displacement ^ chart_infinity)).exp()                   # [] Motor

# A hyperboloid of one sheet, z**2 = x**2 + (y / 0.65)**2 - 1: a waist, and two ends that open towards infinity.
hyperboloid: Quadric = (mv.z * (mv.z & Point) - mv.x * (mv.x & Point) - mv.y * (mv.y & Point) / 0.65**2
                        + chart_infinity * (chart_infinity & Point))                    # [] Sphere <- Point
# Inverted in the unit sphere about (1.7, 0, 0), outside it: infinity goes to that centre, so both ends meet there
# in a conical tip, and the waist becomes the rim of an opening.
inversion = (translator(mv.x * 1.7) >> (chart_origin - chart_infinity)).normalized()   # [] Sphere: a reflection
cyclide: Quadric = inversion >> hyperboloid(inversion << Point)                  # [] Sphere <- Point

# The circle of radius 2.5 about the z axis goes round the hyperboloid's waist; inverted with it, it threads the
# opening crosswise. Normalized before inverting, it stays a unit circle: circle * circle == -1.
sphere = chart_origin - chart_infinity * (2.5**2 / 2)                       # [] Sphere: radius 2.5 about the origin
circle = inversion >> (mv.z ^ sphere).normalized()                          # [] Bivector
# The ring around it: the inverted sphere at unit weight, whose square is the circle's squared radius, dressed as a
# torus of tube 0.02 in the plane z = 0.
sphere = inversion >> sphere
sphere = sphere / -(sphere | chart_infinity)
radius_squared = sphere.squared()
sphere = sphere - chart_infinity * (0.02**2 / 2)
ring: Quadric = sphere * (sphere & Point) + radius_squared * (
    mv.z * (mv.z & Point) - 0.02**2 * chart_infinity * (chart_infinity & Point))      # [] Sphere <- Point

# The flat tracer's camera: its pose carries a camera at the origin, looking along -x with z up, to `position`
# looking at `target`. Moving the scene by the inverse brings that camera to the eye, which also looks along -x; the
# great circles through the eye are the chart's straight lines through its origin, so the view is the flat one.
position, target = mv(Direction, [2.8, -9.0, 4.8]), mv(Direction, [1.0, 0.0, 0.0])
forward = (target - position).normalized()
right = ((forward ^ mv.z) * mv.xyz.inverse()).normalized()                  # z up: right is forward x z
aim = (1 - forward * mv.x).normalized()                                      # -x onto forward
roll = (1 + right * (aim >> mv.y)).normalized()                              # then y onto right
camera = (translator(position) * roll * aim).inverse()                       # [] Motor
narrow: Direction = mv(Direction, pixel_grid(shape, np.radians(44))).normalized()

# One turn around the circle over 36 frames; the ring stays. Per frame both bodies are traced and the image keeps
# the nearer hit in each pixel. The inversion centre is outside the solid, so the cyclide is lit on both sides.
flow = (circle * (np.linspace(0.0, 2 * np.pi, 36, endpoint=False) / 2)).exp()   # [frames] Motor
bodies = (stack((motion >> cyclide(motion << Point), camera >> ring(camera << Point))) for motion in camera * flow)
traced = ((facing.abs(), angle) for facing, angle in (trace(both, narrow) for both in bodies))

colors = np.stack([render.TEAL, render.GOLD])                                # the cyclide, the ring
display(Image(filename=save_animation(render.scene_frames(traced, colors, shape), "cyclides_linked_vortex", 60)))

## 10. The Rest of the Gallery

The other scenes of the flat tracer, each built in the chart about the eye and seen through its camera; `scenarios.py` holds their constructions.

In [ ]:
from examples.geometry.cyclides import scenarios

# One still per scene: rings, a peanut, pinched and two-lobed cyclides, and the six families of the sphere model.
still = ["cyclide", "peanut", "elliptic_ring", "split_ring", "pinched", "six_families", "hyperboloid", "two_lobed"]
images = [next(render.scene_frames(scenarios.flat_scene(name, 1), render.PALETTE, shape)) for name in still]
render.draw_images(images, 4);

In [ ]:
# The scenes with vortices: each part turned once around its own circle.
for name in ["linked_tori", "pinched_vortex"]:
    frames = render.scene_frames(scenarios.flat_scene(name, 36), render.PALETTE, shape)
    display(Image(filename=save_animation(frames, f"cyclides_{name}", 60)))